<a id="inizio"></a>
# 06 · Pandas: Series, DataFrame e import dei dati · Soluzioni

⏱ ~55 min &nbsp;·&nbsp; Dati: `../Dati/TexasTurbine.csv`, `../Dati/letture_pod_2025.csv`, `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/utility.db`, `../Dati/homework/letture_marzo.csv`, `../Dati/homework/clienti.xlsx`, `../Dati/homework/anagrafica.db`

**In questo notebook impariamo a**

- creare Series e DataFrame e accedere ai loro elementi
- leggere file CSV ed Excel con i parametri di lettura, anche più fogli insieme
- caricare una tabella da un database SQL con una query

<a id="indice"></a>
**Indice**

- [1. Importare le librerie](#sez-1)
- [2. Le strutture dati di pandas](#sez-2)
    - [2.1 Series](#sez-2-1)
    - [2.2 DataFrame](#sez-2-2)
- [3. File e cartelle con os](#sez-3)
- [4. File CSV](#sez-4)
    - [4.1 Un CSV salvato da Excel in italiano](#sez-4-1)
- [5. File Excel](#sez-5)
    - [5.1 Più fogli in un file](#sez-5-1)
- [6. Database SQL](#sez-6)
- [7. Extra](#sez-7)
    - [Esercizio 6.1 · Il file del distributore](#es-6-1)
    - [Esercizio 6.2 · La reperibilità](#es-6-2)

<a id="sez-1"></a>
## 1. Importare le librerie

Pandas è una libreria open-source per l'analisi e la manipolazione dei dati in Python. Offre
strutture dati e funzioni pensate per lavorare con dati strutturati, come tabelle e serie temporali.

Per usare una libreria la importiamo con la parola chiave `import`, di solito con un nome abbreviato;
per pandas il nome convenzionale è `pd`. È buona pratica importare tutte le librerie necessarie
all'inizio del notebook, in modo che chi lo legge veda subito di che cosa ha bisogno.

In [ ]:
import pandas as pd

<a id="sez-2"></a>
## 2. Le strutture dati di pandas

Pandas si basa su due strutture dati principali. La **Series** è un array unidimensionale in cui
ogni elemento ha un'etichetta, mentre il **DataFrame** è una struttura bidimensionale, con righe e
colonne etichettate. Un DataFrame si può vedere come un insieme di Series, una per colonna, che
condividono le stesse etichette di riga.

<a id="sez-2-1"></a>
### 2.1 Series

Una **Series** è un array unidimensionale che può contenere qualsiasi tipo di dato, come interi,
stringhe, numeri decimali o oggetti Python. Ogni elemento è associato a un'etichetta, chiamata
**indice**. Nella cella qui sotto creiamo una Series a partire da una lista e le diamo un nome con
`name`; poiché non indichiamo un indice, pandas usa le posizioni 0, 1, 2 e 3.

In [ ]:
# creare una Series da una lista
dati = [10, 20, 30, 40]
s = pd.Series(dati, name="valori")
s

Con il parametro `index` possiamo assegnare agli elementi etichette scelte da noi al posto delle
posizioni numeriche. La lista delle etichette deve avere la stessa lunghezza della lista dei dati.

In [ ]:
s_custom = pd.Series(dati, index=["a", "b", "c", "d"])
s_custom

Agli elementi di una Series si accede con le parentesi quadre, come per gli elementi di una lista.
Nella prima cella qui sotto usiamo l'indice numerico di `s`, nella seconda l'etichetta `"b"` di
`s_custom`, che restituisce il secondo valore.

In [ ]:
# accesso tramite indice numerico
s[0]

In [ ]:
# accesso tramite indice personalizzato
s_custom["b"]

Anche lo slicing funziona come con le liste Python. L'espressione `s[1:3]` restituisce una nuova
Series con gli elementi nelle posizioni 1 e 2, perché l'estremo finale è escluso, e ogni elemento
conserva la sua etichetta originale.

In [ ]:
# slicing della Series
s[1:3]

<a id="sez-2-2"></a>
### 2.2 DataFrame

Un **DataFrame** è una struttura bidimensionale con dati allineati in righe e colonne, simile a un
foglio di calcolo o a una tabella SQL. Un modo semplice per crearlo è partire da un dizionario, in
cui ogni chiave diventa il nome di una colonna e la lista associata ne contiene i valori. Con
`index` diamo un'etichetta anche alle righe, e nel prossimo notebook la useremo per selezionarle.

In [ ]:
# creare un DataFrame da un dizionario
dati = {
    "Nome": ["Alice", "Bob", "Charlie"],
    "Età": [24, 27, 22],
    "Città": ["Roma", "Milano", "Torino"],
}
df = pd.DataFrame(dati, index=["id_1", "id_2", "id_3"])
df

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Hands-on**

Salva in `prime_due` le prime due righe di `df`, con il metodo `head()` visto nel notebook 05, e in
`forma` il suo attributo `shape`. Poiché `df` ha tre righe e tre colonne, `forma` deve valere `(3, 3)`.

*Prova anche:* con `head(1)`.

</div>

In [ ]:
prime_due = df.head(2)
forma = df.shape
print(forma)
prime_due

In [ ]:
# Verifica: esegui senza modificare
assert list(prime_due.index) == ["id_1", "id_2"], "❌ prime_due: le prime due righe si ottengono con df.head(2)"
assert forma == (3, 3), "❌ forma: shape è un attributo, quindi si scrive senza parentesi"
print("✅ Tutto corretto")

<a id="sez-3"></a>
## 3. File e cartelle con os

Il modulo `os` di Python serve a interagire con il sistema operativo, per esempio per navigare tra
le cartelle, gestire i file e ottenere informazioni sul sistema. La funzione `os.listdir()`
restituisce l'elenco dei file e delle cartelle contenuti in una cartella e, se non riceve argomenti,
considera la cartella di lavoro. Nella cella qui sotto stampiamo questo elenco una voce per riga.

In [ ]:
import os

# elencare i file nella cartella di lavoro
elenco_file = os.listdir()
print("Contenuto della cartella:")
for file in elenco_file:
    print(file)

La cartella di lavoro di un notebook è quella in cui si trova il file, e `os.getcwd()` ne
restituisce il percorso completo. I dati del corso stanno nella cartella `Dati`, che si trova un
livello più in alto; poiché `..` indica la cartella superiore, il percorso relativo dei file è
`../Dati/`. Con `os.path.exists()` controlliamo che un file esista prima di leggerlo; la funzione
restituisce `True` se il percorso è corretto e `False` altrimenti.

In [ ]:
os.getcwd()

In [ ]:
os.path.exists("../Dati/TexasTurbine.csv")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Hands-on**

Salva in `file_dati` l'elenco dei file contenuti nella cartella `Dati`, passando a `os.listdir()` il
percorso relativo della cartella. Nell'elenco devono comparire, tra gli altri, `TexasTurbine.csv` e
`utility.db`.

</div>

In [ ]:
file_dati = os.listdir("../Dati")
for file in file_dati:
    print(file)

In [ ]:
# Verifica: esegui senza modificare
assert "TexasTurbine.csv" in file_dati and "utility.db" in file_dati, "❌ file_dati: il percorso relativo della cartella è \"../Dati\""
print("✅ Tutto corretto")

<a id="sez-4"></a>
## 4. File CSV

I file CSV (*Comma-Separated Values*) sono uno dei formati più comuni per archiviare e scambiare dati
tabulari. Ogni riga del file corrisponde a una riga della tabella, e i valori sono separati da un
carattere, di solito la virgola. Pandas li legge con la funzione `read_csv()`, che restituisce un
DataFrame, e con il metodo `head()` ne vediamo le prime cinque righe.

In [ ]:
# leggere un file CSV
df_turbina = pd.read_csv("../Dati/TexasTurbine.csv")
df_turbina.head()

Dopo la lettura conviene controllare che cosa è arrivato. Il metodo `info()` riassume il numero di
righe e di colonne, il tipo di ogni colonna e quanti valori non mancano, mentre `describe()` calcola
le statistiche principali delle colonne numeriche, come media, minimo, massimo e quartili.

In [ ]:
df_turbina.info()

In [ ]:
df_turbina.describe()

Quando un file non segue il formato standard, `read_csv()` si adatta attraverso alcuni parametri.
Quelli che useremo più spesso sono i seguenti:

- `usecols` indica quali colonne leggere dal file;
- `sep` indica il delimitatore tra le colonne, per esempio `,`, `;` oppure `\t`;
- `decimal` indica il separatore dei decimali, per esempio `,` invece di `.`.

<a id="sez-4-1"></a>
### 4.1 Un CSV salvato da Excel in italiano

Il file `letture_pod_2025.csv` contiene i consumi mensili per fascia di sei POD ed è stato esportato
da un Excel configurato in italiano. Se proviamo a leggerlo con i parametri predefiniti, la cella qui
sotto si ferma con un errore, che in questo caso è voluto. L'informazione che serve per capirlo si
trova nell'ultima riga del messaggio.

In [ ]:
pd.read_csv("../Dati/letture_pod_2025.csv")

L'errore è un `UnicodeDecodeError`, e nasce dal fatto che pandas legge il file come UTF-8 e incontra
un byte che in UTF-8 non è valido, cioè la `è` di "Caffè". I file salvati da Excel su Windows usano spesso l'encoding
`latin-1`, che va quindi indicato con il parametro `encoding`. I CSV italiani, inoltre, separano le
colonne con `;` e scrivono i decimali con la virgola, per cui alla lettura servono tre parametri.
Nella seconda cella controlliamo con `dtypes` il tipo di ogni colonna.

In [ ]:
letture = pd.read_csv("../Dati/letture_pod_2025.csv", sep=";", decimal=",", encoding="latin-1")
letture.head()

In [ ]:
letture.dtypes

La colonna `kwh` risulta di tipo `float64`, cioè numerica. Senza `decimal=","` pandas l'avrebbe
letta come testo, e su una colonna di testo non si possono calcolare somme o medie. Il parametro
`encoding="latin-1"`, invece, conviene aggiungerlo solo quando l'errore lo richiede, perché su un
file UTF-8 che contiene lettere accentate non darebbe alcun errore, ma produrrebbe caratteri
sbagliati senza segnalarlo.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Hands-on**

Rileggi `letture_pod_2025.csv` con gli stessi tre parametri e aggiungi `usecols`, in modo da tenere
soltanto le colonne `cliente`, `fascia` e `kwh`. Il DataFrame `letture_ridotte` deve avere 216 righe e
3 colonne.

</div>

In [ ]:
letture_ridotte = pd.read_csv(
    "../Dati/letture_pod_2025.csv", sep=";", decimal=",", encoding="latin-1", usecols=["cliente", "fascia", "kwh"]
)
letture_ridotte.head()

In [ ]:
# Verifica: esegui senza modificare
assert list(letture_ridotte.columns) == ["cliente", "fascia", "kwh"], "❌ letture_ridotte: usecols riceve la lista delle tre colonne"
assert letture_ridotte.shape == (216, 3), "❌ letture_ridotte: le righe sono 216, come nel file"
print("✅ Tutto corretto")

<a id="sez-5"></a>
## 5. File Excel

I file Excel sono molto usati per i dati strutturati e, a differenza dei CSV, possono contenere più
fogli. Pandas li legge con la funzione `read_excel()`, che senza altre indicazioni carica il primo
foglio. Come esempio leggiamo il carico elettrico della zona Nord nel 2024, che Terna pubblica con
un valore ogni quarto d'ora.

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico.head()

<a id="sez-5-1"></a>
### 5.1 Più fogli in un file

Con `pd.ExcelWriter` possiamo scrivere più DataFrame nello stesso file Excel, ciascuno nel proprio
foglio. Per vederlo creiamo prima due piccoli DataFrame di esempio, con le stesse colonne.

In [ ]:
df1 = pd.DataFrame({
    "ID": [1, 2, 3],
    "Nome": ["Alice", "Bob", "Charlie"],
    "Età": [25, 30, 35],
})
df2 = pd.DataFrame({
    "ID": [4, 5, 6],
    "Nome": ["David", "Eva", "Frank"],
    "Età": [40, 45, 50],
})

L'istruzione `with pd.ExcelWriter("dati.xlsx") as writer` apre il file in scrittura e lo chiude alla
fine del blocco. Dentro il blocco il metodo `to_excel()` scrive ogni DataFrame nel foglio indicato
da `sheet_name`, e `index=False` evita di salvare l'indice come colonna aggiuntiva.

In [ ]:
# creare un file Excel con più fogli
with pd.ExcelWriter("dati.xlsx") as writer:
    df1.to_excel(writer, sheet_name="Foglio1", index=False)
    df2.to_excel(writer, sheet_name="Foglio2", index=False)
print("File 'dati.xlsx' creato.")

Per leggere un foglio specifico si passa il suo nome al parametro `sheet_name` di `read_excel()`.
Con `dtypes` controlliamo poi che le colonne siano arrivate con il tipo giusto, per esempio `Età`
come numero intero.

In [ ]:
# leggere un foglio specifico
df_foglio1 = pd.read_excel("dati.xlsx", sheet_name="Foglio1")
df_foglio1.dtypes

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Hands-on**

Leggi in `df_foglio2` il secondo foglio di `dati.xlsx`, indicandone il nome con `sheet_name`. La
colonna `Nome` deve contenere David, Eva e Frank.

</div>

In [ ]:
df_foglio2 = pd.read_excel("dati.xlsx", sheet_name="Foglio2")
df_foglio2

In [ ]:
# Verifica: esegui senza modificare
assert list(df_foglio2["Nome"]) == ["David", "Eva", "Frank"], "❌ df_foglio2: il secondo foglio si chiama \"Foglio2\""
print("✅ Tutto corretto")

Con `sheet_name=None` pandas legge tutti i fogli in una volta e restituisce un dizionario, in cui la
chiave è il nome del foglio e il valore è il DataFrame corrispondente. Le chiavi del dizionario ci
mostrano quali fogli sono stati letti.

In [ ]:
# leggere tutti i fogli dal file Excel
fogli = pd.read_excel("dati.xlsx", sheet_name=None)
fogli.keys()

Poiché i due fogli hanno le stesse colonne, possiamo metterli uno sotto l'altro con `pd.concat()`,
a cui passiamo i DataFrame contenuti nel dizionario. Con `ignore_index=True` l'indice del risultato
riparte da 0, invece di ripetere gli indici dei singoli fogli.

In [ ]:
# concatenare i DataFrame
df_excel_unito = pd.concat(fogli.values(), ignore_index=True)
df_excel_unito

<a id="sez-6"></a>
## 6. Database SQL

Spesso i dati stanno in un database, organizzati in tabelle che si interrogano con il linguaggio
SQL. Python comunica con il database attraverso una connessione, e la funzione `pd.read_sql` esegue
una query e ne restituisce il risultato come DataFrame. In questo notebook usiamo SQLite, un database
contenuto in un singolo file, `utility.db`, che si apre con il modulo `sqlite3` della libreria
standard. La prima query chiede al database l'elenco delle sue tabelle.

In [ ]:
import sqlite3

con = sqlite3.connect("../Dati/utility.db")
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)

La tabella `sqlite_master` è quella in cui SQLite tiene l'elenco del proprio contenuto, e dal
risultato vediamo che il database contiene le tabelle `clienti`, `pod` e `letture`. Per leggere una
tabella intera si usa una query come `SELECT * FROM letture`, in cui l'asterisco indica tutte le
colonne; poiché la query non pone condizioni, arrivano anche tutte le righe.

In [ ]:
letture_db = pd.read_sql("SELECT * FROM letture", con)
letture_db.head()

Quando ci serve solo una parte dei dati, la clausola `WHERE` aggiunge una condizione alla query. In
questo modo il filtro lo esegue il database, e in Python arrivano soltanto le righe che servono;
nell'esempio teniamo i POD con una potenza di almeno 30 kW.

In [ ]:
pd.read_sql("SELECT * FROM pod WHERE potenza_kw >= 30", con)

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Hands-on**

Cambia la condizione del `WHERE` in modo da leggere in `pod_piccoli` i POD con una potenza inferiore a
10 kW. Le righe sono 9, tutte con 3 o 6 kW.

*Prova anche:* con `potenza_kw = 50`, che dà 5 righe.

</div>

In [ ]:
pod_piccoli = pd.read_sql("SELECT * FROM pod WHERE potenza_kw < 10", con)
pod_piccoli

In [ ]:
# Verifica: esegui senza modificare
assert len(pod_piccoli) == 9, "❌ pod_piccoli: la condizione è potenza_kw < 10"
print("✅ Tutto corretto")

Al termine del lavoro la connessione si chiude con il metodo `close()`, che libera il file del
database. Dopo la chiusura, per eseguire altre query occorre aprire una nuova connessione con
`sqlite3.connect()`.

In [ ]:
con.close()

Riassunto in una pagina: [Da Excel a pandas](../Schede/Scheda_Excel_pandas.md).

<a id="sez-7"></a>
## 7. Extra

Questa sezione raccoglie esercizi brevi sui punti principali del notebook, cioè la lettura di un CSV
italiano, di un foglio Excel e di una tabella SQL, da fare a casa per ripassare. Durante la lezione la
saltiamo, e ogni esercizio ha la sua verifica, che conferma il risultato o dice che cosa controllare.

<a id="es-6-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 6.1 · Il file del distributore**

Lavoriamo nell'ufficio consumi di una piccola società di vendita di energia, e il distributore ci manda
il CSV con le letture giornaliere di marzo degli otto POD dei nostri clienti, scritto all'italiana con
il punto e virgola tra le colonne e la virgola nei decimali. Un'officina di Lecco segnala una bolletta
di marzo tripla del solito, quindi leggiamo il file con pandas e controlliamo con `info()` e
`describe()` che i consumi siano arrivati come numeri e quale sia la lettura più alta. La tabella ha 248
righe e tre colonne, `kwh` è di tipo `float64` e il massimo è di 1530.3 kWh, molto sopra i 250 kWh di
una giornata normale.

<details>
<summary>Passo per passo</summary>

1. Leggi `../Dati/homework/letture_marzo.csv` in `letture`, indicando a `read_csv` il separatore delle
   colonne e quello dei decimali.
2. Le istruzioni già scritte sotto la lettura mostrano i tipi delle colonne con `info()`, le prime righe
   con `head()` e le statistiche con `describe()`. Controlla che `kwh` sia `float64`, perché un numero
   letto come testo non si può sommare né confrontare con altri numeri, e leggi il massimo nella riga
   `max` di `describe()`.

</details>

<details>
<summary>Suggerimento 1</summary>

Un CSV salvato da Excel in italiano usa un separatore di colonne e un separatore decimale diversi
da quelli standard, e `read_csv` ha un parametro per ciascuno.

</details>

<details>
<summary>Suggerimento 2</summary>

Servono `sep=";"` e `decimal=","`; se `kwh` risulta letta come testo, di solito manca il
`decimal`.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 6.1**

`info()` mostra subito se `kwh` è stata letta come numero, perché in quel caso il suo tipo è `float64`.
Questo file si legge senza indicare `encoding`, a differenza di `letture_pod_2025.csv`, e il massimo
di 1530.3 kWh, lontano dalla media di circa 77 kWh, è la lettura da segnalare al distributore.

</div>

In [ ]:
letture = pd.read_csv("../Dati/homework/letture_marzo.csv", sep=";", decimal=",")
letture.info()
print(letture.head())
print(letture.describe())

In [ ]:
from corso import verifica
verifica("6.1")

<a id="es-6-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 6.2 · La reperibilità**

Siamo di reperibilità nel fine settimana e l'Officina Meccanica Fumagalli, che contesta la bolletta, chiede
la potenza del suo contatore e i prezzi per fascia che le applichiamo. Il gestionale è spento, ma la potenza
sta nella tabella `pod` del database dell'anagrafica e i prezzi nel foglio Listino dell'Excel dei clienti, e
con pandas leggiamo da lì soltanto quello che serve, in poche righe che tornano utili alla prossima
reperibilità. Il contatore dell'officina ha 30 kW e i prezzi sono di 0.21, 0.19 e 0.17 euro/kWh nelle
fasce F1, F2 e F3.

<details>
<summary>Passo per passo</summary>

1. Leggi in `listino` il foglio `Listino` di `../Dati/homework/clienti.xlsx`, indicandolo con `sheet_name`, perché senza
   questo parametro `read_excel` legge il primo foglio, che è l'anagrafica dei clienti.
2. Apri `../Dati/homework/anagrafica.db` con `sqlite3.connect` e leggi con `pd.read_sql` in `officina` la sola riga dell'officina
   della tabella `pod`, con un `WHERE` sulla colonna `cliente`. La riga `con.close()`, già scritta
   nella cella, chiude poi la connessione, che a quel punto non serve più.

</details>

<details>
<summary>Suggerimento 1</summary>

Con `read_excel` il foglio si sceglie con `sheet_name`; con il database si apre la connessione, si
legge con `read_sql` e si chiude la connessione.

</details>

<details>
<summary>Suggerimento 2</summary>

In SQL il testo va tra apici singoli: `WHERE cliente = 'Officina Meccanica Fumagalli'`.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 6.2**

Il filtro con `WHERE` viene eseguito dal database, che restituisce a pandas soltanto la riga richiesta.
Con una tabella di otto righe la differenza non si nota, ma su tabelle grandi evita di caricare in
memoria dati che non servono.

</div>

In [ ]:
import sqlite3

listino = pd.read_excel("../Dati/homework/clienti.xlsx", sheet_name="Listino")

con = sqlite3.connect("../Dati/homework/anagrafica.db")
officina = pd.read_sql("SELECT * FROM pod WHERE cliente = 'Officina Meccanica Fumagalli'", con)
con.close()

print(listino)
officina

In [ ]:
from corso import verifica
verifica("6.2")

---

**Fine del notebook 06.** Precedente: [05 · Oggetti ed errori](05_Oggetti_ed_errori.ipynb) &nbsp;·&nbsp; Prossimo: [E2 · Esercitazione 2](Esercitazione_2.ipynb)

<sub>Blocco 2 · Controllo del flusso, funzioni e primi dati in pandas · Giornata 1 · Aula Base</sub>